<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


## Key steps


This Noebook contains the following tasks:
- **Step 1:** Mark all launch sites on a map
- **Step 2:** Mark the success/failed launches for each site on the map
- **Step 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, we should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [6]:
#!pip3 install folium
#!pip3 install wget
#!pip install --upgrade folium
#!pip3 install pandas

In [1]:
import folium
import wget
import pandas as pd

In [2]:
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

## Step 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [3]:
# Download and read the `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

100% [................................................................................] 7710 / 7710

In [4]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. So, Let's visualize those locations by pinning them on a map.

We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [10]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [11]:
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

Now, let's add a circle for each launch site in data frame `launch_sites`


In [13]:
# Coordinates for each launch site
coordinates = [
    [28.562302, -80.577356],  # CCAFS LC-40
    [28.563197, -80.576820],  # CCAFS SLC-40
    [28.573255, -80.646895],  # KSC LC-39A
    [34.632834, -120.610745]  # VAFB SLC-4E
]

# Launch site names
site_names = [
    "CCAFS LC-40",
    "CCAFS SLC-40",
    "KSC LC-39A",
    "VAFB SLC-4E"
]

# Add a circle and marker for each launch site
for coordinate, name in zip(coordinates, site_names):
    # Create a circle with a popup label showing the site name
    circle = folium.Circle(
        coordinate,
        radius=1000,
        color='#000000',
        fill=True
    ).add_child(folium.Popup(name))
    
    # Create a marker with a text label
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12; color:#d35400;"><b>{name}</b></div>'
        )
    )
    
    # Add the circle and marker to the map
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display the map
site_map

**Proximity to Equator**
Florida sites (CCAFS SLC-40, CCAFS LC-40, KSC LC-39A) closer to Equator → better efficiency for most launches.
California site (VAFB SLC-4E) farther north → mainly for polar/sun-synchronous orbits.

**Proximity to Coast**
All sites near coastlines → launch paths over open water for safety and lower risk to populated areas.


# Step 2: Mark the success/failed launches for each site on the map


In [5]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [16]:
marker_cluster = MarkerCluster()

In [17]:
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [18]:
site_map.add_child(marker_cluster)

for index, row in spacex_df.iterrows():
    lat = float(row['Lat'])
    lon = float(row['Long'])
    color = row['marker_color']          # 'green' or 'red'
    label = f"{row['Launch Site']} | class={'Success' if row['class'] == 1 else 'Failure'}"
    
    folium.Marker(
        location=[lat, lon],
        popup=label,
        icon=folium.Icon(color='white', icon_color=color)
    ).add_to(marker_cluster)

site_map

From the color-labeled markers in marker clusters, we should be able to easily identify which launch sites have relatively high success rates.


# Step 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while we are exploring the map, we can easily find the coordinates of any points of interests (such as railway)


In [19]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move the mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [14]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

Markred down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [25]:
# find coordinate of the closet coastline
launch_site_lat = 28.563197
launch_site_lon = -80.576820

# coastline point I read from the map
coastline_lat = 28.56359
coastline_lon = -80.56807

distance_coastline = calculate_distance(launch_site_lat, launch_site_lat, coastline_lat, coastline_lon)

After obtained its coordinate, created a `folium.Marker` to show the distance


In [28]:
distance_marker = folium.Marker(
    location=[coastline_lat,coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )
    )
distance_marker.add_to(site_map)

Drawed a `PolyLine` between a launch site to the selected coastline point


In [31]:
# draw a line between the launch site and the coastline point
lines = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
    weight=2,
)
site_map.add_child(lines)

# show the map
site_map


Your updated map with distance line should look like the following screenshot:


we will draw a line betwee a launch site to its closest city, railway, highway, etc. we need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [15]:
# Launch site coordinates
launch_site_lat = 28.563197
launch_site_lon = -80.576820

# Coordinates of nearby points of interest
coordinates = [
    [28.56349, -80.57084],  # Railway
    [28.56352, -80.58689],  # Highway
    [28.6118, -80.80816],   # Titusville (city)
]

# Names of the points of interest
site_names = [
    "Railway",
    "Highway",
    "Titusville",
]


In [29]:
# Add the launch site marker
folium.Marker(
    [launch_site_lat, launch_site_lon],
    popup="Launch Site",
    icon=folium.Icon(color='red', icon='rocket')
).add_to(site_map)

# Add markers for each point of interest and draw lines
for i, (coord, name) in enumerate(zip(coordinates, site_names)):
    # Calculate distance
    distance = calculate_distance(launch_site_lat, launch_site_lon, coord[0], coord[1])
    
    # Create marker with distance label
    distance_marker = folium.Marker(
        location=coord,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 18PX; color:#00FF00; font-weight: bold;"><b>{distance:.2f} KM</b></div>'
        )
    )
    distance_marker.add_to(site_map)
    
    # Draw a line between the launch site and the point of interest
    line = folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], coord],
        weight=2,
        color='blue'
    )
    site_map.add_child(line)

In [30]:
folium.Marker(
        coord,
        popup=name,
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(site_map)

# Show the map
site_map